# 4.8 — Probabilistic Forecasting and Uncertainty

v30-nll: Gaussian NLL, predicted uncertainty (σ̂), calibration,
and uncertainty maps.

**Terminology:**
- **σ̂** = predicted Gaussian standard deviation (model output)
- **σ_e** = std(ŷ − y), the prediction-error standard deviation
  (observed property of the residuals)

These are distinct quantities. σ̂ is what the model *thinks* its
uncertainty is; σ_e is how variable its errors actually are.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

import torch
# v30-nll is the only model with log_var
d_nll = C.load_dump("v30-nll", "mr0.00")
P_nll = d_nll["preds"]                    # (M, K, N, 5)
T_nll = d_nll["targets"][:, :, :, :NV]
M_nll = d_nll["masks"][:, :, :, :NV]
LV    = d_nll["log_var"]                   # (M, K, N, 5)
print(f"v30-nll dump: M={P_nll.shape[0]}, K={P_nll.shape[1]}, N={P_nll.shape[2]}")

p = P_nll.numpy().astype(np.float64)
t = T_nll.numpy().astype(np.float64)
m = (M_nll.numpy() > 0.5) & KEEP[None, None]
lv = LV.numpy().astype(np.float64)
sigma_hat = np.sqrt(np.exp(lv))  # predicted σ̂

## Per-lead Gaussian NLL

In [ ]:
eps2 = (p - t) ** 2
nll = 0.5 * (eps2 * np.exp(-lv) + lv)

nll_per_lead = np.zeros((K, NV))
cnt_per_lead = np.zeros((K, NV))
for ki in range(K):
    for vi in range(NV):
        mk = m[:, ki, :, vi]
        nll_per_lead[ki, vi] = nll[:, ki, :, vi][mk].sum()
        cnt_per_lead[ki, vi] = mk.sum()
nll_mean = nll_per_lead / np.maximum(cnt_per_lead, 1)

fig, axes = plt.subplots(1, NV, figsize=(17, 3.4))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    ax.plot(range(1, K), nll_mean[1:, vi], "o-", ms=4, lw=1.4, color="#E1A730")
    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("Mean Gaussian NLL [normalised units]")
fig.suptitle("Probabilistic MAE: Gaussian NLL vs lead time (MR=0)", y=1.04)
plt.tight_layout(); C.save_fig(fig, "48_nll_vs_lead"); plt.show()

## Calibration: observed coverage vs predicted intervals

For a well-calibrated Gaussian, the fraction of observations within
±kσ̂ should match the theoretical CDF. Points above the diagonal =
overconfident (intervals too narrow); below = underconfident.

In [ ]:
from scipy.stats import norm as sp_norm
k_vals = np.linspace(0.5, 3.0, 11)
expected = 2 * sp_norm.cdf(k_vals) - 1   # two-sided coverage

z = np.abs(p - t) / np.maximum(sigma_hat, 1e-8)  # |standardised residual|

fig, axes = plt.subplots(1, NV, figsize=(17, 3.4))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    observed = []
    for kk in k_vals:
        inside = (z[:, :, :, vi] <= kk) & m[:, :, :, vi]
        observed.append(inside.sum() / max(m[:, :, :, vi].sum(), 1))
    ax.plot(expected, observed, "o-", ms=4, lw=1.3, color="#E1A730", label="Probabilistic MAE")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect calibration")
    ax.set_xlabel("Expected coverage"); ax.set_title(v, fontsize=10)
    ax.grid(alpha=.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].set_ylabel("Observed coverage")
axes[-1].legend(fontsize=8)
fig.suptitle("Calibration of the Probabilistic MAE (MR=0), all lead times and stations: observed vs expected fraction of residuals inside ±kσ, k = 0.5…3", y=1.04)
plt.tight_layout(); C.save_fig(fig, "48_calibration"); plt.show()

## Predicted uncertainty (σ̂) vs actual MAE — sharpness check

Bin observations by predicted-σ̂ quintiles. A sharp model should show
actual MAE tracking predicted σ̂ monotonically.

In [ ]:
REF_KI = 6
fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    mk = m[:, REF_KI, :, vi]
    actual_mae = np.abs(p[:, REF_KI, :, vi] - t[:, REF_KI, :, vi])
    pred_sig = sigma_hat[:, REF_KI, :, vi]
    valid = mk.ravel()
    ae = actual_mae.ravel()[valid]
    ps = pred_sig.ravel()[valid]
    q = np.percentile(ps, [20, 40, 60, 80])
    bins_s = np.digitize(ps, q)
    means_ae = [ae[bins_s == b].mean() for b in range(5)]
    means_ps = [ps[bins_s == b].mean() for b in range(5)]
    ax.bar(range(5), means_ae, color="#1F5F6B", alpha=0.7, label="actual MAE")
    ax.plot(range(5), means_ps, "D-", color="#E1A730", ms=6, lw=1.5,
            label="predicted σ̂")
    ax.set_xticks(range(5))
    ax.set_xticklabels(["Q1\n(low σ̂)", "Q2", "Q3", "Q4", "Q5\n(high σ̂)"], fontsize=7)
    ax.set_title(v, fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel(f"value at {LEAD[REF_KI]}")
axes[-1].legend(fontsize=7)
fig.suptitle(f"Sharpness: actual MAE vs predicted σ̂ quintiles at {LEAD[REF_KI]}", y=1.04)
plt.tight_layout(); C.save_fig(fig, "48_sharpness"); plt.show()

## Per-station predicted uncertainty (σ̂) maps

In [ ]:
# Mean predicted σ̂ per station per lead
sigma_station = np.zeros((K, P_nll.shape[2], NV))
for ki in range(K):
    for vi in range(NV):
        mk = m[:, ki, :, vi]
        s = (sigma_hat[:, ki, :, vi] * mk).sum(0)
        c = mk.sum(0)
        sigma_station[ki, :, vi] = np.where(c > 0, s / np.maximum(c, 1), np.nan)

fig, axes = plt.subplots(NV, 3, figsize=(16, 3.5 * NV))
for vi, v in enumerate(VARS):
    for ci, (ki, klbl) in enumerate([(1, "+30 min"), (4, "+2 h"), (12, "+6 h")]):
        ax = axes[vi, ci]
        vals = sigma_station[ki, :, vi]
        valid = ~np.isnan(vals)
        sc = ax.scatter(stn.longitude[valid], stn.latitude[valid],
                        c=vals[valid], s=30, cmap="inferno",
                        edgecolors="k", linewidths=0.3)
        fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
        ax.set_aspect("equal"); ax.tick_params(labelsize=5)
        ax.set_xlim(5.8, 10.6); ax.set_ylim(45.8, 47.9)
        if vi == 0: ax.set_title(klbl, fontsize=11, fontweight="bold")
        if ci == 0: ax.set_ylabel(f"{v} [{C.UNITS[v]}]", fontsize=9)
fig.suptitle("Probabilistic MAE: mean predicted σ̂ per station (normalised units)", y=1.02)
plt.tight_layout(); C.save_fig(fig, "48_uncertainty_maps"); plt.show()

## Per-station σ̂ vs actual MAE

The most important diagnostic: *does the model correctly identify
which stations are hard to forecast?*

In [ ]:
from scipy.stats import pearsonr
REF_KI = 6
fig, axes = plt.subplots(1, NV, figsize=(17, 3.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    # per-station mean σ̂ and MAE at REF_KI
    mk = m[:, REF_KI, :, vi]
    ae = np.abs(p[:, REF_KI, :, vi] - t[:, REF_KI, :, vi])
    mae_stn = np.where(mk.sum(0) > 0, (ae * mk).sum(0) / np.maximum(mk.sum(0), 1), np.nan)
    sig_stn = sigma_station[REF_KI, :, vi]
    valid = ~np.isnan(mae_stn) & ~np.isnan(sig_stn)
    ax.scatter(sig_stn[valid], mae_stn[valid], s=15, alpha=0.6, color="#E1A730")
    r, pv = pearsonr(sig_stn[valid], mae_stn[valid])
    ax.set_title(f"{v}  (r={r:.2f}, p={pv:.1e})", fontsize=9)
    ax.set_xlabel("mean predicted σ̂", fontsize=8)
    ax.grid(alpha=.3)
    # reference line
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, "k--", lw=0.8, alpha=0.5)
axes[0].set_ylabel(f"actual MAE at {LEAD[REF_KI]}")
fig.suptitle(f"Per-station: predicted σ̂ vs actual MAE at {LEAD[REF_KI]} (MR=0)", y=1.04)
plt.tight_layout(); C.save_fig(fig, "48_sigma_vs_mae"); plt.show()

del d_nll, P_nll, T_nll, M_nll, LV  # free memory

## Interpretation

**NLL** should increase with lead time as prediction becomes harder.

**Calibration:** points above the diagonal mean the model is
overconfident (predicted intervals too narrow); below = underconfident.

**Sharpness:** if actual MAE tracks predicted σ̂ across quintiles,
the model correctly adjusts its confidence to match difficulty.

**σ̂ vs MAE scatter:** a high Pearson r means the model identifies
which stations are intrinsically harder to forecast. This is the
most operationally useful property of the probabilistic head.

Compare NLL-trained v30 vs Huber-trained v27 on point-forecast MAE
(notebook 4.1) to assess whether the NLL objective helps or hurts
point accuracy.